In [2]:
# Core Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Tuning
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)  # suppress per-trial noise

# Models and Metrics
import xgboost as xgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

# Visualization configuration
sns.set_theme(style='whitegrid')

# Warnings
import warnings
warnings.filterwarnings('ignore')

print('All imports OK!')

All imports OK!


In [19]:
try:
    # Try Kaggle paths first
    train = pd.read_csv("/kaggle/input/competitions/playground-series-s6e3/train.csv")
    test = pd.read_csv("/kaggle/input/competitions/playground-series-s6e3/test.csv")
except:
    # Fallback to local paths if the above fails for any reason
    train = pd.read_csv("data/train.csv")
    test = pd.read_csv("data/test.csv")

train['Churn'] = (train['Churn'] == 'Yes').astype(int)

train['SeniorCitizen'] = train['SeniorCitizen'].astype('category')
test['SeniorCitizen']  = test['SeniorCitizen'].astype('category')

cat_cols = train.select_dtypes('object').columns
train[cat_cols] = train[cat_cols].astype('category')

print('Train shape:', train.shape)
print('Test shape: ', test.shape)

Train shape: (594194, 21)
Test shape:  (254655, 20)


In [20]:
train.head()

,id,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,0,Male,0,Yes,Yes,29,Yes,No,DSL,Yes,...,Yes,Yes,No,No,One year,Yes,Mailed check,60.10,1653.85,0
1,1,Male,0,Yes,Yes,58,Yes,No,DSL,Yes,...,No,Yes,Yes,No,Two year,No,Credit card (automatic),69.50,3778.20,0
2,2,Male,0,Yes,No,58,Yes,Yes,Fiber optic,No,...,No,No,Yes,Yes,Month-to-month,Yes,Electronic check,100.40,5841.35,0
3,3,Female,0,No,No,1,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,69.70,70.70,1
4,4,Female,0,No,No,1,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.45,70.45,1


## 1. Feature Engineering

In [ ]:
categorical_cols = [
    'gender', 'SeniorCitizen', 'Partner', 'Dependents',
    'PhoneService', 'MultipleLines', 'InternetService',
    'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
    'TechSupport', 'StreamingTV', 'StreamingMovies',
    'Contract', 'PaperlessBilling', 'PaymentMethod'
]
numerical_cols   = ['tenure', 'MonthlyCharges', 'TotalCharges']

INTERNET_ADDONS  = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
                    'TechSupport', 'StreamingTV', 'StreamingMovies']

def engineer_static_features(df):
    df = df.copy()

    # Layer 1: Base
    df['tenure_bin'] = pd.cut(df['tenure'], bins=[-1,10,20,30,40,60,np.inf],
                              labels=['0-10mo','10-20mo','20-30mo','30-40mo','40-60mo','60mo+'])
    df['avg_monthly_charge']  = df['TotalCharges'] / df['tenure']
    df['charge_diff']         = df['MonthlyCharges'] - df['avg_monthly_charge']
    df['num_services']        = (df[INTERNET_ADDONS].isin(['Yes'])).sum(axis=1)
    df['has_internet']        = (df['InternetService'] != 'No').astype(int)
    df['services_per_dollar'] = df['num_services'] / df['MonthlyCharges']
    df['contract_strength']   = df['Contract'].map({'Month-to-month':0,'One year':1,'Two year':2}).astype(int)
    df['SeniorCitizen_int']   = df['SeniorCitizen'].astype(int)
    df['high_risk']    = ((df['Contract']=='Month-to-month') & (df['InternetService']=='Fiber optic')).astype(int)
    df['digital_only'] = ((df['PaperlessBilling']=='Yes') & (df['PaymentMethod']=='Electronic check')).astype(int)

    # Layer 2: Economic
    df['Price_Shock']            = df['MonthlyCharges'] - (df['TotalCharges'] / df['tenure'])
    df['Price_Shock_Pct']        = df['Price_Shock'] / (df['TotalCharges'] / df['tenure'])
    df['Cost_Per_Service']       = df['MonthlyCharges'] / df['num_services']
    df['Has_Service']            = (df['num_services'] > 0).astype(int)
    df['monthly_to_total_ratio'] = df['MonthlyCharges'] / df['TotalCharges']
    df['charge_per_tenure_year'] = (df['MonthlyCharges'] * 12) / df['tenure']
    df['expected_total']         = df['tenure'] * df['MonthlyCharges']
    df['total_charge_residual']  = df['TotalCharges'] - df['expected_total']

    # Layer 3: Behavioral
    df['Household_Size_Proxy'] = (df['Partner']=='Yes').astype(int) + (df['Dependents']=='Yes').astype(int)
    df['Is_Automatic_Payment'] = df['PaymentMethod'].str.contains('automatic', case=False, na=False).astype(int)
    df['Tenure_Squared']       = df['tenure'] ** 2

    # Layer 4: MultipleLines 3-way split
    df['phone_multiple'] = (df['MultipleLines'] == 'Yes').astype(int)
    df['phone_none']     = (df['MultipleLines'] == 'No phone service').astype(int)

    # Layer 5: Missed add-ons
    has_internet_mask = df['InternetService'] != 'No'
    df['missed_addons'] = 0
    for col in INTERNET_ADDONS:
        df.loc[has_internet_mask, 'missed_addons'] += (df.loc[has_internet_mask, col] == 'No').astype(int)
    df['internet_no_addons']  = (has_internet_mask & (df['num_services'] == 0)).astype(int)
    df['addon_adoption_rate'] = np.where(df['num_services'] > 0,
                                         df['num_services'] / (df['num_services'] + df['missed_addons']), 0)

    # Layer 6: Tenure x Contract
    df['new_mtm']           = ((df['tenure'] <= 20) & (df['Contract'] == 'Month-to-month')).astype(int)
    df['tenure_x_contract'] = df['tenure'] * df['contract_strength']
    df['loyal_but_mtm']     = ((df['tenure'] > 30) & (df['Contract'] == 'Month-to-month')).astype(int)

    # Layer 7: Streaming vs protection
    df['streaming_count']  = ((df['StreamingTV']=='Yes').astype(int) + (df['StreamingMovies']=='Yes').astype(int))
    df['protection_count'] = ((df['OnlineSecurity']=='Yes').astype(int) + (df['OnlineBackup']=='Yes').astype(int) +
                               (df['DeviceProtection']=='Yes').astype(int) + (df['TechSupport']=='Yes').astype(int))
    df['only_streaming']   = ((df['streaming_count'] > 0) & (df['protection_count'] == 0)).astype(int)
    df['streaming_ratio']  = df['streaming_count'] / df['num_services'].replace(0, 1)

    # Layer 8: Fiber risk
    df['fiber_no_support']       = ((df['InternetService']=='Fiber optic') &
                                    (df['TechSupport']=='No') & (df['OnlineSecurity']=='No')).astype(int)
    df['fiber_protection_score'] = (df['InternetService']=='Fiber optic').astype(int) * df['protection_count']

    # Layer 9: Individual service binary flags
    for svc in INTERNET_ADDONS:
        df[f'{svc}_bin']         = (df[svc] == 'Yes').astype(int)
        df[f'{svc}_no_internet'] = (df[svc] == 'No internet service').astype(int)

    # Layer 10: MTM risk tiers
    df['mtm_risk_tier'] = 0
    is_mtm = df['Contract'] == 'Month-to-month'
    df.loc[is_mtm & (df['tenure'] <= 20),            'mtm_risk_tier'] = 3
    df.loc[is_mtm & df['tenure'].between(21, 30),     'mtm_risk_tier'] = 2
    df.loc[is_mtm & (df['tenure'] > 30),              'mtm_risk_tier'] = 1

    # Layer 11: Senior digital combo
    df['senior_digital'] = ((df['SeniorCitizen']==1) & (df['PaperlessBilling']=='Yes') &
                             (df['PaymentMethod']=='Electronic check')).astype(int)

    # Layer 12: Price-shocked MTM
    df['price_shocked_mtm'] = ((df['Price_Shock'] > 0) & (df['Contract'] == 'Month-to-month')).astype(int)

    # Layer 13: Expensive fiber with no support
    median_charge = df['MonthlyCharges'].median()
    df['fiber_expensive_no_support'] = ((df['InternetService']=='Fiber optic') &
                                         (df['MonthlyCharges'] > median_charge) &
                                         (df['TechSupport']=='No')).astype(int)

    # Interaction string columns for target encoding
    df['Contract_x_PaymentMethod']  = df['Contract'].astype(str) + '_' + df['PaymentMethod'].astype(str)
    df['Contract_x_TechSupport']    = df['Contract'].astype(str) + '_' + df['TechSupport'].astype(str)
    df['Internet_x_Security']       = df['InternetService'].astype(str) + '_' + df['OnlineSecurity'].astype(str)
    df['Internet_x_TechSupport']    = df['InternetService'].astype(str) + '_' + df['TechSupport'].astype(str)
    df['Senior_x_Contract']         = df['SeniorCitizen'].astype(str) + '_' + df['Contract'].astype(str)
    df['tenure_bin'] = df['tenure_bin'].astype(str)
    return df

train_static = engineer_static_features(train)
test_static  = engineer_static_features(test)
print(f'Static features shape: {train_static.shape}')

Static features shape: (594194, 78)


,id,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,StreamingMovies_no_internet,mtm_risk_tier,senior_digital,price_shocked_mtm,fiber_expensive_no_support,Contract_x_PaymentMethod,Contract_x_TechSupport,Internet_x_Security,Internet_x_TechSupport,Senior_x_Contract
0,0,Male,0,Yes,Yes,29,Yes,No,DSL,Yes,...,0,0,0,0,0,One year_Mailed check,One year_Yes,DSL_Yes,DSL_Yes,0_One year
1,1,Male,0,Yes,Yes,58,Yes,No,DSL,Yes,...,0,0,0,0,0,Two year_Credit card (automatic),Two year_Yes,DSL_Yes,DSL_Yes,0_Two year
2,2,Male,0,Yes,No,58,Yes,Yes,Fiber optic,No,...,0,1,0,0,1,Month-to-month_Electronic check,Month-to-month_No,Fiber optic_No,Fiber optic_No,0_Month-to-month
3,3,Female,0,No,No,1,Yes,No,Fiber optic,No,...,0,3,0,0,0,Month-to-month_Electronic check,Month-to-month_No,Fiber optic_No,Fiber optic_No,0_Month-to-month
4,4,Female,0,No,No,1,Yes,No,Fiber optic,No,...,0,3,0,0,0,Month-to-month_Electronic check,Month-to-month_No,Fiber optic_No,Fiber optic_No,0_Month-to-month


## 2. Prepare Fold-Encoded Data

We pre-build the target-encoded arrays for all 5 folds **once**, outside Optuna.
This means each trial just trains XGBoost on already-encoded data — making trials fast.

In [ ]:
interaction_cols = [
    'Contract_x_PaymentMethod', 'Contract_x_TechSupport',
    'Internet_x_Security', 'Internet_x_TechSupport', 'Senior_x_Contract', 'tenure_bin'
]
all_categorical_cols = categorical_cols + interaction_cols

numeric_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
feature_cols = numeric_cols + all_categorical_cols

X_raw      = train_static[feature_cols].copy()
y          = train_static['Churn'].values
X_test_raw = test_static[feature_cols].copy()

N_SPLITS = 5
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

# Pre-encode each fold and store as numpy arrays
# This avoids repeating string ops inside every Optuna trial
fold_data   = []   # list of (X_tr, y_tr, X_val, y_val)
test_arrays = []   # one test encoding per fold (averaged at submission time)

for fold, (train_idx, val_idx) in enumerate(skf.split(X_raw, y), 1):
    X_tr_df  = X_raw.iloc[train_idx].copy()
    X_val_df = X_raw.iloc[val_idx].copy()
    X_tst_df = X_test_raw.copy()
    y_tr     = y[train_idx]
    y_val    = y[val_idx]

    global_mean = y_tr.mean()
    y_tr_s = pd.Series(y_tr, index=X_tr_df.index)

    for col in all_categorical_cols:
        X_tr_df[col]  = X_tr_df[col].astype(str)
        X_val_df[col] = X_val_df[col].astype(str)
        X_tst_df[col] = X_tst_df[col].astype(str)
        means = y_tr_s.groupby(X_tr_df[col]).mean()
        X_tr_df[col]  = X_tr_df[col].map(means).fillna(global_mean).astype(float)
        X_val_df[col] = X_val_df[col].map(means).fillna(global_mean).astype(float)
        X_tst_df[col] = X_tst_df[col].map(means).fillna(global_mean).astype(float)

    fold_data.append((X_tr_df.values, y_tr, X_val_df.values, y_val, val_idx))
    test_arrays.append(X_tst_df.values)
    print(f'Fold {fold} encoded: X_tr={X_tr_df.shape}, X_val={X_val_df.shape}')

feature_names = X_raw.columns.tolist()
print(f'\nTotal features: {len(feature_names)}')
print('Fold data pre-encoded and ready for Optuna.')

## 3. Optuna Hyperparameter Search

### How this works:
- Each **trial** proposes a set of hyperparameters using Tree-structured Parzen Estimator (TPE)
- TPE learns from previous trials — it focuses search on promising regions, unlike grid/random search
- Each trial runs a full **5-fold stratified CV** and returns mean OOF AUC
- Optuna maximises AUC, pruning bad trials early via **MedianPruner**

**`N_TRIALS = 100`** is a good balance for Kaggle GPU notebooks (~15–30 min).
Increase to 200 if you have time.

In [ ]:
N_TRIALS      = 30    # Slashed to save time!
EARLY_STOP    = 100   
TUNE_N_EST    = 1000  

def objective(trial):
    params = dict(
        objective             = 'binary:logistic',
        eval_metric           = 'auc',
        n_estimators          = TUNE_N_EST,
        early_stopping_rounds = EARLY_STOP,  
        use_label_encoder     = False,
        random_state          = 42,
        verbosity             = 0,
        device                = 'cuda',   
        tree_method           = 'hist',   
        
        # ── The Heavy Hitters ───────────────────────────────────────────
        max_depth             = trial.suggest_int('max_depth', 3, 10),
        learning_rate         = trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
    )

    fold_aucs = []
    for X_tr, y_tr, X_val, y_val, _ in fold_data:
        model = xgb.XGBClassifier(**params)
        model.fit(
            X_tr, y_tr,
            eval_set=[(X_val, y_val)],
            verbose=False,
        )
        preds = model.predict_proba(X_val)[:, 1]
        fold_aucs.append(roc_auc_score(y_val, preds))

    return np.mean(fold_aucs)

# ── This is the part that prints your outputs! ──────────────────────

def print_callback(study, trial):
    print(f"Trial {trial.number:>3} | AUC: {trial.value:.5f} | Best: {study.best_value:.5f}")

study = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=42)
)

# Run the 30 trials and trigger the print_callback every time
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=False, callbacks=[print_callback])

print(f'\nBest OOF AUC : {study.best_value:.5f}')
print(f'Best params  :')
for k, v in study.best_params.items():
    print(f'  {k}: {v}')

## 4. Optuna Visualisation

In [ ]:
from optuna.visualization.matplotlib import plot_optimization_history, plot_param_importances

# Plot 1: History
plot_optimization_history(study)
plt.title('Optimization History', fontsize=13)
plt.show()

# Plot 2: Importances
plot_param_importances(study)
plt.show()

## 5. Final KFold Training with Best Params

Now we retrain the full CV loop using the Optuna-tuned hyperparameters,
collecting OOF predictions and tracking best iterations for the full-data refit.

In [ ]:
best_params = study.best_params.copy()
best_params.update({
    'objective':         'binary:logistic',
    'eval_metric':       'auc',
    'n_estimators':      2000,
    'use_label_encoder': False,
    'random_state':      42,
    'verbosity':         0,
    'device':            'cuda',   # ← add this
    'tree_method':       'hist',   # ← and this
})

oof_preds  = np.zeros(len(y))
test_preds = np.zeros(len(test_arrays[0]))
fold_aucs  = []
best_iters = []

for fold, (X_tr, y_tr, X_val, y_val, val_idx) in enumerate(fold_data, 1):
    model = xgb.XGBClassifier(**best_params)
    model.fit(X_tr, y_tr,
              eval_set=[(X_val, y_val)],
              early_stopping_rounds=50,
              verbose=False)

    oof_preds[val_idx] = model.predict_proba(X_val)[:, 1]
    test_preds        += model.predict_proba(test_arrays[fold-1])[:, 1] / N_SPLITS
    best_iters.append(model.best_iteration)

    fold_auc = roc_auc_score(y_val, oof_preds[val_idx])
    fold_aucs.append(fold_auc)
    print(f'Fold {fold}  |  AUC: {fold_auc:.5f}  |  Best iter: {model.best_iteration}')

overall_auc   = roc_auc_score(y, oof_preds)
avg_best_iter = int(round(np.mean(best_iters)))
print(f'\nOOF AUC     : {overall_auc:.5f}')
print(f'Mean CV AUC : {np.mean(fold_aucs):.5f}  +-  {np.std(fold_aucs):.5f}')
print(f'Avg best iter (for refit): {avg_best_iter}')

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

class ResidualBlock(nn.Module):
    def __init__(self, dim, dropout=0.2):
        super().__init__()
        self.block = nn.Sequential(
            nn.Linear(dim, dim),
            nn.BatchNorm1d(dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim, dim),
            nn.BatchNorm1d(dim)
        )
        self.activation = nn.GELU()

    def forward(self, x):
        return self.activation(x + self.block(x))  # skip connection

class AdvancedMLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()

        # Input projection (wider)
        self.input_layer = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.BatchNorm1d(256),
            nn.GELU()
        )

        # Residual blocks
        self.res_blocks = nn.Sequential(
            ResidualBlock(256, dropout=0.3),
            ResidualBlock(256, dropout=0.3),
            ResidualBlock(256, dropout=0.2)
        )

        # Downstream compression
        self.head = nn.Sequential(
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.GELU(),
            nn.Dropout(0.2),

            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.GELU(),

            nn.Linear(64, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        x = self.input_layer(x)
        x = self.res_blocks(x)
        x = self.head(x)
        return x

# 2. Containers
oof_preds_nn = np.zeros(len(y))
test_preds_nn = np.zeros(len(test_arrays[0]))
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"Using device: {device}")

# 3. Training Loop (reusing fold_data from cell 2)
for fold, (X_tr, y_tr, X_val, y_val, val_idx) in enumerate(fold_data, 1):
    # Scaling
    scaler = StandardScaler()
    X_tr_scaled = scaler.fit_transform(X_tr)
    X_val_scaled = scaler.transform(X_val)
    X_tst_scaled = scaler.transform(test_arrays[fold-1])
    
    # To Tensors
    X_tr_t = torch.FloatTensor(X_tr_scaled).to(device)
    y_tr_t = torch.FloatTensor(y_tr).reshape(-1, 1).to(device)
    X_val_t = torch.FloatTensor(X_val_scaled).to(device)
    y_val_t = torch.FloatTensor(y_val).reshape(-1, 1).to(device)
    X_tst_t = torch.FloatTensor(X_tst_scaled).to(device)
    
    # Datasets
    train_ds = TensorDataset(X_tr_t, y_tr_t)
    train_loader = DataLoader(train_ds, batch_size=1024, shuffle=True)
    
    # Model, Loss, Optimizer
    model = AdvancedMLP(X_tr.shape[1]).to(device)
    criterion = nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    
    # Early Stopping params
    best_val_auc = 0
    patience = 10
    counter = 0
    best_state = None
    
    # Training
    for epoch in range(100):
        model.train()
        for batch_X, batch_y in train_loader:
            optimizer.zero_grad()
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
            
        # Validation
        model.eval()
        with torch.no_grad():
            val_preds = model(X_val_t).cpu().numpy()
            val_auc = roc_auc_score(y_val, val_preds)
            
        if val_auc > best_val_auc:
            best_val_auc = val_auc
            best_state = model.state_dict()
            counter = 0
        else:
            counter += 1
            
        if counter >= patience:
            break
            
    # Load best model
    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        oof_preds_nn[val_idx] = model(X_val_t).cpu().numpy().flatten()
        test_preds_nn += model(X_tst_t).cpu().numpy().flatten() / N_SPLITS
        
    print(f"Fold {fold} | NN AUC: {best_val_auc:.5f}")

# 4. Correlation Analysis (XGBoost vs NN)
correlation = np.corrcoef(oof_preds, oof_preds_nn)[0, 1]
overall_nn_auc = roc_auc_score(y, oof_preds_nn)

print("\n" + "=" * 30)
print(f"XGBoost OOF AUC : {overall_auc:.5f}")
print(f"NN OOF AUC      : {overall_nn_auc:.5f}")
print(f"Prediction Correlation: {correlation:.4f}")
print("=" * 30)

# 5. Visualization
plt.figure(figsize=(10, 6))
sns.regplot(
    x=oof_preds,
    y=oof_preds_nn,
    scatter_kws={'alpha': 0.1, 's': 1},
    line_kws={'color': 'red'}
)
plt.xlabel("XGBoost OOF Predictions")
plt.ylabel("NN OOF Predictions")
plt.title(f"Model Prediction Correlation (r = {correlation:.4f})")
plt.show()

if correlation < 0.98:
    print("Good news! Correlation is sufficiently low. Stacking/ensemble should improve the score.")
else:
    print("Correlation is very high (>0.98). A simple ensemble may not improve performance much.")

## 6. Full-Data Refit

Train one final model on **100% of train** using the tuned params.
Iteration count is scaled up by `5/4` since each CV fold only saw 80% of data.

In [ ]:
full_data_iters = int(round(avg_best_iter * N_SPLITS / (N_SPLITS - 1)))
print(f'CV avg best iter: {avg_best_iter}  ->  Full-data iters: {full_data_iters}')

# Target-encode on full train
X_full      = X_raw.copy()
X_test_full = X_test_raw.copy()
global_mean_full = y.mean()
y_full_s = pd.Series(y, index=X_full.index)

for col in all_categorical_cols:
    X_full[col]      = X_full[col].astype(str)
    X_test_full[col] = X_test_full[col].astype(str)
    means = y_full_s.groupby(X_full[col]).mean()
    X_full[col]      = X_full[col].map(means).fillna(global_mean_full).astype(float)
    X_test_full[col] = X_test_full[col].map(means).fillna(global_mean_full).astype(float)

refit_params = best_params.copy()
refit_params['n_estimators'] = full_data_iters

final_model = xgb.XGBClassifier(**refit_params)
final_model.fit(X_full.values, y, verbose=False)

test_preds_refit = final_model.predict_proba(X_test_full.values)[:, 1]
print(f'Full-data model trained. Predicted churn rate: {test_preds_refit.mean():.4f}')

## 7. Feature Importance

In [ ]:
fi = pd.Series(final_model.feature_importances_, index=feature_names).sort_values(ascending=True).tail(40)
fig, ax = plt.subplots(figsize=(10, 11))
fi.plot.barh(ax=ax, color='steelblue', edgecolor='white')
ax.set_title('Feature Importance — Top 40 (Optuna-tuned full-data model)', fontsize=13, fontweight='bold')
ax.set_xlabel('F-score')
plt.tight_layout(); plt.show()

## 8. Submissions

Two files. **Submit the refit one first** — it uses 100% of train data with tuned params.

In [ ]:
# 8. Ensemble OOF and Test Predictions (XGBoost + NN)
best_weight = 0.0
best_auc = 0.0

for w in np.linspace(0, 1, 101):
    ensemble_oof = w * oof_preds + (1 - w) * oof_preds_nn
    auc = roc_auc_score(y, ensemble_oof)
    if auc > best_auc:
        best_auc = auc
        best_weight = w

print(f"Best Weight: {best_weight:.2f} XGB / {1-best_weight:.2f} NN")
print(f"XGB OOF AUC     : {overall_auc:.5f}")
print(f"NN OOF AUC      : {overall_nn_auc:.5f}")
print(f"Ensemble OOF AUC: {best_auc:.5f} (Improvement: {best_auc - max(overall_auc, overall_nn_auc):.5f})")

test_preds_ensemble = best_weight * test_preds + (1 - best_weight) * test_preds_nn
pd.DataFrame({'id': test_static['id'], 'Churn': test_preds_ensemble}).to_csv('submission_ensemble.csv', index=False)
print("Saved submission_ensemble.csv")

In [ ]:
pd.DataFrame({'id': test_static['id'], 'Churn': test_preds}).to_csv('submission_v9_cv.csv', index=False)
pd.DataFrame({'id': test_static['id'], 'Churn': test_preds_refit}).to_csv('submission_v9_refit.csv', index=False)

print('Saved:')
print('  submission_v9_cv.csv    (5-fold CV average)')
print('  submission_v9_refit.csv (full-data refit  <- submit this first)')
print(f'\nCV    predicted churn rate : {test_preds.mean():.4f}')
print(f'Refit predicted churn rate : {test_preds_refit.mean():.4f}')
print(f'Original churn rate        : {train_static["Churn"].mean():.4f}')